In [ ]:
# Resolve paths consistently from either the repository root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Identifying and fixing missing reports of crime in USA

In [ ]:
#sets the notebook width
from IPython.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))
%load_ext autoreload
%autoreload 2

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Standard Libraries
import sys
import os
from pathlib import Path
import glob
import math

# Third party libraries
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

import statsmodels.api as sm
# from pymc3 import Normal, Model, Uniform,Poisson,Deterministic
# from pymc3 import traceplot,forestplot,summary
# from pymc3 import glm, Model, Metropolis, NUTS, sample
# import pymc3 as pm

# figures
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

# Local
# from modules import ste

In [ ]:
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

def stdrcparams(usetex=False):
    """
    Set several mpl.rcParams and sns.set_style for my taste.
    ----
    usetex = True
    ----
    """
    sns.set_style("white")
    sns.set_style({"xtick.direction": "in",
                 "ytick.direction": "in"})
    rcparams = {'text.usetex': usetex,
              'font.family': 'sans-serif',
              'font.sans-serif': ['Times'],
#               'text.latex.unicode': True,
#               'text.latex.preamble': [r"\usepackage[T1]{fontenc}",
#                                       r"\usepackage{lmodern}",
#                                       r"\usepackage{amsmath}",
#                                       r"\usepackage{mathptmx}"
#                                       ],
              'axes.labelsize': 30,
              'axes.titlesize': 30,
              'axes.spines.left':True,
              'axes.spines.bottom':True,
              'axes.spines.top': False,
              'axes.spines.right':False,
              'ytick.left': 'on',
              'xtick.bottom': 'on',
              'xtick.labelsize': '25',
              'ytick.labelsize': '25',
              'axes.linewidth': 1.8,
              'xtick.major.width': 1.8,
              'xtick.minor.width': 1.8,
              'xtick.major.size': 14,
              'xtick.minor.size': 7,
              'xtick.major.pad': 10,
              'xtick.minor.pad': 10,
              'ytick.major.width': 1.8,
              'ytick.minor.width': 1.8,
              'ytick.major.size': 14,
              'ytick.minor.size': 7,
              'ytick.major.pad': 10,
              'ytick.minor.pad': 10,
              'axes.labelpad': 15,
              'axes.titlepad': 15,
              "xtick.direction":
                "out",
              "ytick.direction": "out"
              }
    mpl.rcParams.update(rcparams)

mpl.rcParams['lines.linewidth'] = 5
mpl.rcParams['lines.color'] = '#3690c0'

In [ ]:
stdrcparams(usetex=True)
figsize=stdfigsize(ratio=-1)
xs,ys=figsize
figsize

# Crime data

In [ ]:
crime_df=pd.read_csv('../data/clean_dataframes/crime.csv',index_col=0)
print(crime_df.shape)

crime_df=crime_df[~np.isnan(crime_df['FIPS'])]
print(crime_df.shape)

crime_df['FIPS']=crime_df['FIPS'].astype(int)
crime_df.head()


In [ ]:
crime_df.Year.unique()

In [ ]:
df_sample = pd.read_csv('../data/laucnty16.csv',index_col=0)
df_sample['State FIPS Code'] = df_sample['State FIPS Code'].apply(lambda x: str(x).zfill(2))
df_sample['County FIPS Code'] = df_sample['County FIPS Code'].apply(lambda x: str(x).zfill(3))
df_sample['FIPS'] = df_sample['State FIPS Code'] + df_sample['County FIPS Code']
df_sample['State']=[item[-1] for item in df_sample['County Name/State Abbreviation'].str.split(', ')]
df_sample['County']=[item[0] for item in df_sample['County Name/State Abbreviation'].str.split(', ')]
df_sample=df_sample[['State FIPS Code','County FIPS Code','County','State','FIPS']]
df_sample.head()

In [ ]:
state_fips=dict(np.transpose([df_sample['State'].values,df_sample['State FIPS Code'].values]))

crime_df['State FIPS']=crime_df['State'].map(state_fips)

crime_df.head()

## Identifying missing report by agency

In [ ]:
crime_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
agency_data=crime_df[(crime_df.ORI=='ILCPD00')] # ILCPD00= Chicago Police Department
plt.plot(agency_data.Year,agency_data.Burglary)
ax.set_ylabel(r"{}".format('Burglary'))
ax.set_xlabel(r"Year")

In [ ]:
def plot_crime(x,y,ax,color,crime=None):
    plt.plot(x,y,color=color)
    if crime==None:
        ax.set_ylabel(r"{}".format('Murder'))
    else:
        ax.set_ylabel(r"{}".format(crime))
    ax.set_xlabel(r"Year")

In [ ]:
def fill_miss_report_nan(crime_df,variable='Murder',sigma=-1.6448):
    crime_no_corrections=crime_df.copy()
    for ori in np.unique(crime_no_corrections.ORI):
        df=crime_no_corrections[(crime_no_corrections.ORI==ori)]
        y=np.array(df[variable])
        years=df[((y-np.mean(y))/np.std(y)<sigma)&(y==0)].Year.values
        indices=df[((y-np.mean(y))/np.std(y)<sigma)&(y==0)].index
        if len(indices)>0:
            print(ori)
            for index in indices:
                crime_no_corrections.at[index,variable]=np.nan
    crime_nan=crime_no_corrections
    return crime_nan
   
def interpolate_by_ori(crime_nan,variable='Murder'):   
    crime_fixed=[]
    for ori in np.unique(crime_nan.ORI):
        df=crime_nan[crime_nan['ORI']==ori]
        df[variable]=df[variable].interpolate(kind='quadratic').ffill().bfill()
        crime_fixed.append(df)

    crime_fixed=pd.concat(crime_fixed)
    crime_fixed=crime_fixed.reset_index(drop=True)
    return crime_fixed

In [ ]:
crime_nan=fill_miss_report_nan(crime_df,variable='Murder',sigma=-1.6448)
crime_fixed=interpolate_by_ori(crime_nan,variable='Murder')

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
agency_data=crime_df[(crime_df.ORI=='AL00301')] # ILCPD00= Chicago Police Department
plt.plot(agency_data.Year,agency_data.Murder)
agency_data=crime_fixed[(crime_fixed.ORI=='AL00301')]
plt.plot(agency_data.Year,agency_data.Murder)
ax.set_ylabel(r"{}".format('Murder'))
ax.set_xlabel(r"Year")

In [ ]:
crime_fixed_murders=crime_fixed.copy()

In [ ]:
crime_df=crime_fixed.copy()

# Aggregate data by county

In [ ]:
crime_df.head()

In [ ]:
county_agg=crime_df.groupby(['County','FIPS','State','Year']).sum().reset_index()
county_agg['County']=county_agg['County'].str.replace(' COUNTY','')
county_agg=county_agg.groupby(['County','FIPS','State','Year']).sum().reset_index()
county_agg.head()

In [ ]:
county_agg[(county_agg.County=='Milwaukee')&(county_agg.State=='WI')].head()

In [ ]:
county_agg["County"] = county_agg["County"].map(str) +', '+ county_agg["State"]

In [ ]:
crime_county=county_agg[['County', 'FIPS', 'State', 'Year', 'Murder', 
                         'Robbery', 'Assault', 'Burglary', 'Larceny','Motor Vehicle Theft']].copy()
crime_county.head()

In [ ]:
crime_county[crime_county.State=='NB']=crime_county[crime_county.State=='NB'].replace('NB','NE')

In [ ]:
crime_county['FIPS']=crime_county['FIPS'].astype(str)
crime_county.head()

In [ ]:
crime_county.to_csv('../data/clean_dataframes/crime_county_data.csv')